# Notebook 4: Live Demo Toolkit

**Purpose**: This is your **speed toolkit** for the Customer Scenarios interview. Every cell is copy-paste ready. When the timer starts, grab what you need and adapt it.

**How to use this notebook**:
- Before the interview, skim it so you know what's here
- During the interview, copy the relevant template and modify it
- The patterns here cover 90% of what you'll be asked to build

**Sections**:
- 4.1 Universal Starter Template
- 4.2 System Prompt Templates
- 4.3 Few-Shot Example Templates
- 4.4 Quick Eval Pattern
- 4.5 Structured Output Patterns
- 4.6 Prompt Chaining Pattern
- 4.7 When Things Go Wrong -- Recovery Patterns
- 4.8 The Demo Presentation Flow
- 4.9 Production Talking Points
- Summary: Speed Run Checklist

---

## Setup

In [ ]:
!pip install anthropic pydantic -q
import anthropic
import json

# Colab: use Secrets (key icon in sidebar)
# from google.colab import userdata
# client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))
client = anthropic.Anthropic()
print("Client ready.")

---

## 4.1 The Universal Starter Template

Copy this first. It works for any scenario. You can always add complexity later, but you want something calling the API in the first 60 seconds.

In [ ]:
# === THE ONE FUNCTION YOU ALWAYS NEED ===

def run(user_message, system_prompt="You are a helpful assistant.", model="claude-haiku-4-5-20251001"):
    """Simple single-turn call. Returns the text response."""
    response = client.messages.create(
        model=model, max_tokens=4096,
        system=system_prompt,
        messages=[{"role": "user", "content": user_message}]
    )
    return response.content[0].text

# Quick smoke test
print(run("Say 'toolkit ready' and nothing else."))

In [ ]:
# === MULTI-TURN VERSION ===
# Use this when the scenario needs conversation history (e.g., customer support bot, tutor)

def run_conversation(messages, system_prompt="You are a helpful assistant.", model="claude-haiku-4-5-20251001"):
    """Multi-turn call. Pass full message history. Returns text response."""
    response = client.messages.create(
        model=model, max_tokens=4096,
        system=system_prompt,
        messages=messages
    )
    return response.content[0].text

# Example: 2-turn conversation
history = [
    {"role": "user", "content": "My name is Jordan."},
    {"role": "assistant", "content": "Nice to meet you, Jordan! How can I help you today?"},
    {"role": "user", "content": "What's my name?"}
]
print(run_conversation(history))

In [ ]:
# === FULL RESPONSE VERSION ===
# Use this when you need stop_reason, usage stats, or multiple content blocks

def run_full(user_message, system_prompt="You are a helpful assistant.", model="claude-haiku-4-5-20251001"):
    """Returns the full response object for inspection."""
    response = client.messages.create(
        model=model, max_tokens=4096,
        system=system_prompt,
        messages=[{"role": "user", "content": user_message}]
    )
    return response

# Example: inspect the full response
resp = run_full("Hello!")
print(f"Model: {resp.model}")
print(f"Stop reason: {resp.stop_reason}")
print(f"Input tokens: {resp.usage.input_tokens}")
print(f"Output tokens: {resp.usage.output_tokens}")
print(f"Text: {resp.content[0].text}")

---

## 4.2 System Prompt Templates

Four battle-tested skeletons. Pick the one that fits your scenario, fill in the blanks, and you have a working system prompt in under 2 minutes.

### Template A: The Classifier

Use when the task is: *sort inputs into categories* (e.g., support tickets, content moderation, sentiment analysis, intent detection).

In [ ]:
# === TEMPLATE A: THE CLASSIFIER ===

classifier_system_prompt = """You are an expert customer support classifier. Your job is to categorize incoming support tickets into one of these categories:

- billing: Payment issues, charges, refunds, subscription changes
- technical: Bugs, errors, crashes, performance problems
- account: Login issues, password resets, profile changes, account deletion
- feature_request: Suggestions for new features or improvements
- general: Questions, feedback, or anything that doesn't fit the above

For each support ticket:
1. Read the message carefully
2. Consider which category best fits the primary issue
3. Return your classification in this exact format:

<classification>
<category>CATEGORY_NAME</category>
<confidence>high/medium/low</confidence>
<reasoning>1-2 sentence explanation of why this category fits.</reasoning>
</classification>

Important rules:
- Choose exactly ONE category
- If multiple categories apply, pick the PRIMARY issue
- Use "low" confidence when the ticket is ambiguous
- Do NOT include any text outside the XML tags"""

# Test it
test_ticket = "I was charged twice for my pro subscription last month. Can you refund the duplicate charge? Also the app crashes when I try to view my invoice."
result = run(test_ticket, system_prompt=classifier_system_prompt)
print(result)

### Template B: The Extractor

Use when the task is: *pull structured data from unstructured text* (e.g., extract entities from emails, parse resumes, extract order details).

In [ ]:
# === TEMPLATE B: THE EXTRACTOR ===

extractor_system_prompt = """You are an expert data extraction system. Your job is to extract structured information from unstructured text.

From each input, extract the following fields:
- person_name: Full name of the person mentioned
- company: Company or organization name
- email: Email address if present
- phone: Phone number if present
- request_type: What the person is asking for
- urgency: high, medium, or low based on language and context

Return your extraction in this exact format:

<extraction>
<person_name>NAME or UNKNOWN</person_name>
<company>COMPANY or UNKNOWN</company>
<email>EMAIL or UNKNOWN</email>
<phone>PHONE or UNKNOWN</phone>
<request_type>DESCRIPTION</request_type>
<urgency>high/medium/low</urgency>
</extraction>

Important rules:
- Use UNKNOWN for any field you cannot determine from the text
- Do NOT guess or hallucinate information that isn't in the input
- Extract information exactly as written (don't reformat phone numbers, etc.)
- Do NOT include any text outside the XML tags"""

# Test it
test_email = """Hi there,

My name is Sarah Chen from Acme Corp. We're having a critical issue with the API integration — our production system has been down for 2 hours. Please call me ASAP at (555) 123-4567 or email sarah.chen@acmecorp.com.

This is urgent.

Thanks,
Sarah"""

result = run(test_email, system_prompt=extractor_system_prompt)
print(result)

### Template C: The Generator

Use when the task is: *create content based on specifications* (e.g., marketing copy, email responses, documentation, reports).

In [ ]:
# === TEMPLATE C: THE GENERATOR ===

generator_system_prompt = """You are an expert customer communication writer. Your job is to generate professional email responses to customer inquiries.

When writing responses, follow these specifications:

Tone: Professional but warm. Empathetic without being overly casual.
Audience: B2B customers — technical decision-makers at mid-size companies.
Length: 3-5 sentences for the main body. Keep it concise and actionable.
Format: Use this structure:

<response>
<subject>EMAIL_SUBJECT_LINE</subject>
<body>
GREETING

MAIN_BODY (acknowledge their issue, provide solution/next steps)

CLOSING
SIGNATURE
</body>
</response>

Important rules:
- Always acknowledge the customer's concern first
- Provide a clear next step or resolution
- Include a specific timeline when possible (e.g., "within 24 hours")
- Sign as "The Support Team"
- Do NOT make promises about features or timelines you can't guarantee
- Do NOT include any text outside the XML tags"""

# Test it
test_inquiry = "We've been waiting 3 days for our API keys and our launch is tomorrow. This is really frustrating — we chose your platform specifically because of the fast onboarding promise."
result = run(test_inquiry, system_prompt=generator_system_prompt)
print(result)

### Template D: The Analyst/Summarizer

Use when the task is: *analyze or summarize content* (e.g., meeting notes, document summaries, competitive analysis, risk assessment).

In [ ]:
# === TEMPLATE D: THE ANALYST/SUMMARIZER ===

analyst_system_prompt = """You are an expert business analyst. Your job is to analyze text and produce structured summaries with actionable insights.

For each input, produce an analysis with these sections:

<analysis>
<summary>2-3 sentence high-level summary of the key points.</summary>
<key_findings>
- Finding 1
- Finding 2
- Finding 3
</key_findings>
<sentiment>positive/negative/neutral/mixed</sentiment>
<action_items>
- Action 1
- Action 2
</action_items>
<risk_level>high/medium/low</risk_level>
<risk_explanation>Why this risk level was assigned.</risk_explanation>
</analysis>

Important rules:
- Base your analysis ONLY on information present in the input
- Be specific in action items — who should do what
- Flag any gaps or missing information
- Do NOT include any text outside the XML tags"""

# Test it
test_report = """Q3 Customer Feedback Summary:

We received 1,247 feedback submissions this quarter. NPS score dropped from 72 to 61. 
Top complaints: (1) API response times increased 40% after the v3 migration, 
(2) documentation is outdated for 12 of 30 endpoints, (3) billing portal 
doesn't support multi-currency for EU customers.

Positive themes: customers love the new dashboard UI, the webhook reliability 
is praised consistently, and enterprise onboarding time dropped from 2 weeks to 3 days.

Three enterprise accounts (ARR > $200K each) mentioned evaluating competitors 
due to API performance issues."""

result = run(test_report, system_prompt=analyst_system_prompt)
print(result)

---

## 4.3 Few-Shot Example Templates

Few-shot examples are the fastest way to improve output quality. Use this pattern when:
- The output format is specific or unusual
- You need consistent behavior across inputs
- The task has subtle nuances that are hard to describe in words

### The XML Pattern for Few-Shot Examples

Put examples in the **user message** (not the system prompt) so they're close to the actual input.

In [ ]:
# === FEW-SHOT TEMPLATE ===

few_shot_system_prompt = """You are a sentiment classifier for product reviews. Classify each review as positive, negative, or neutral. Return your answer in the exact XML format shown in the examples."""

def build_few_shot_prompt(actual_input):
    """Build a few-shot prompt with examples + the actual input."""
    return f"""Here are examples of the expected input and output:

<examples>
<example>
<input>This product is amazing! Best purchase I've made all year. The battery lasts forever and the design is sleek.</input>
<output>
<sentiment>positive</sentiment>
<reasoning>Strong positive language ("amazing", "best"), highlights multiple positive features.</reasoning>
</output>
</example>

<example>
<input>Broke after 2 days. Complete waste of money. Customer service was unhelpful and rude.</input>
<output>
<sentiment>negative</sentiment>
<reasoning>Product failure, negative value assessment ("waste of money"), negative service experience.</reasoning>
</output>
</example>

<example>
<input>It works fine I guess. Does what it says on the box. Nothing special but nothing wrong either.</input>
<output>
<sentiment>neutral</sentiment>
<reasoning>Lukewarm language ("fine I guess", "nothing special"), no strong positive or negative signals.</reasoning>
</output>
</example>
</examples>

Now classify this review:
<input>{actual_input}</input>"""

# Test with different inputs
test_reviews = [
    "The camera quality is decent for the price, but the software is buggy and crashes a lot. Mixed feelings overall.",
    "I can't believe how good this is for $20. Absolutely blown away.",
    "Returned it immediately. The product description was completely misleading."
]

for review in test_reviews:
    result = run(
        build_few_shot_prompt(review),
        system_prompt=few_shot_system_prompt
    )
    print(f"Review: {review[:60]}...")
    print(f"Result: {result}")
    print("-" * 60)

### Pro tip: Few-shot examples in the system prompt vs. user message

| Location | When to use | Benefit |
|----------|-------------|--------|
| System prompt | Same examples for ALL inputs | Benefits from prompt caching (90% cost reduction) |
| User message | Examples vary per request | More flexible, closer to input |

For the interview, putting examples in the user message is simpler and works great.

---

## 4.4 Quick Eval Pattern

This is your **demo function**. When you're presenting to the "customer," you run this to show your solution working across multiple inputs. It prints clean, readable output.

In [ ]:
# === THE QUICK EVAL FUNCTION ===

def quick_eval(system_prompt, test_cases, parse_fn=None, model="claude-haiku-4-5-20251001"):
    """
    Run test cases and display results cleanly.
    
    system_prompt: The system prompt to test
    test_cases: list of dicts with 'input' and optionally 'expected'
    parse_fn: optional function to parse/format the output
    model: model to use
    """
    print(f"Running {len(test_cases)} test cases...\n")
    print("=" * 60)
    results = []
    for i, case in enumerate(test_cases, 1):
        response = client.messages.create(
            model=model,
            max_tokens=2048,
            system=system_prompt,
            messages=[{"role": "user", "content": case["input"]}]
        )
        raw_result = response.content[0].text
        result = parse_fn(raw_result) if parse_fn else raw_result
        
        print(f"Test {i}:")
        print(f"  Input:    {case['input'][:80]}{'...' if len(case['input']) > 80 else ''}")
        if 'expected' in case:
            match = '  ' if 'expected' not in case else ('PASS' if case['expected'].lower() in result.lower() else 'FAIL')
            print(f"  Expected: {case['expected']}  [{match}]")
        print(f"  Output:   {result[:200]}{'...' if len(result) > 200 else ''}")
        print("-" * 60)
        results.append({"input": case["input"], "output": result, "raw": raw_result})
    
    return results

In [ ]:
# === DEMO: Quick eval with the classifier template ===

import re

def parse_category(text):
    """Extract just the category from XML classification output."""
    match = re.search(r'<category>(.*?)</category>', text)
    return match.group(1) if match else text[:100]

test_cases = [
    {
        "input": "I was charged $49.99 but my plan is the $29.99 tier. Please fix this.",
        "expected": "billing"
    },
    {
        "input": "The app crashes every time I try to upload a file larger than 10MB.",
        "expected": "technical"
    },
    {
        "input": "I forgot my password and the reset email never arrives.",
        "expected": "account"
    },
    {
        "input": "It would be great if you added dark mode to the dashboard.",
        "expected": "feature_request"
    },
    {
        "input": "I love your product! Just wanted to say thanks to the team.",
        "expected": "general"
    }
]

results = quick_eval(classifier_system_prompt, test_cases, parse_fn=parse_category)

---

## 4.5 Structured Output Patterns

Three ways to get structured, parseable output from Claude. Use the simplest one that works for your scenario.

### Pattern 1: XML Tags in Prompt

**When to use**: Most cases. Simple, reliable, easy to parse.

**Advantage**: Claude follows XML formatting very reliably. No extra dependencies.

In [ ]:
# === PATTERN 1: XML TAGS ===

xml_system_prompt = """Analyze the given product review and return your analysis in this exact XML format:

<review_analysis>
<sentiment>positive/negative/neutral/mixed</sentiment>
<stars>1-5</stars>
<key_topics>
<topic>TOPIC_1</topic>
<topic>TOPIC_2</topic>
</key_topics>
<summary>One sentence summary.</summary>
</review_analysis>

Do NOT include any text outside the XML tags."""

# Helper to parse XML output
def parse_xml_tag(text, tag):
    """Extract content of a single XML tag. Works for simple (non-nested) tags."""
    match = re.search(f'<{tag}>(.*?)</{tag}>', text, re.DOTALL)
    return match.group(1).strip() if match else None

def parse_xml_tags(text, tag):
    """Extract all instances of a repeated tag."""
    return re.findall(f'<{tag}>(.*?)</{tag}>', text, re.DOTALL)

# Test it
review = "The laptop is incredibly fast and the screen is gorgeous. But the keyboard feels cheap and the trackpad is too small. For the price, I expected better build quality."

raw = run(review, system_prompt=xml_system_prompt)
print("Raw output:")
print(raw)
print("\nParsed:")
print(f"  Sentiment: {parse_xml_tag(raw, 'sentiment')}")
print(f"  Stars: {parse_xml_tag(raw, 'stars')}")
print(f"  Topics: {parse_xml_tags(raw, 'topic')}")
print(f"  Summary: {parse_xml_tag(raw, 'summary')}")

### Pattern 2: JSON Instruction

**When to use**: When the consumer of the output is code (API endpoint, pipeline).

**Advantage**: Directly parseable with `json.loads()`. Good when you need nested data.

In [ ]:
# === PATTERN 2: JSON INSTRUCTION ===

json_system_prompt = """Analyze the given product review and return your analysis as a JSON object with these exact keys:

{
  "sentiment": "positive" | "negative" | "neutral" | "mixed",
  "stars": 1-5,
  "key_topics": ["topic1", "topic2"],
  "summary": "One sentence summary.",
  "would_recommend": true | false
}

Return ONLY the JSON object. No markdown formatting, no explanation, no code fences."""

# Test it
review = "Absolutely love this blender! Makes perfect smoothies every time. Easy to clean too. Only downside is it's a bit loud."

raw = run(review, system_prompt=json_system_prompt)
print("Raw output:")
print(raw)

# Parse it
try:
    parsed = json.loads(raw)
    print("\nParsed JSON:")
    for key, value in parsed.items():
        print(f"  {key}: {value}")
except json.JSONDecodeError as e:
    # Fallback: try stripping markdown code fences
    cleaned = re.sub(r'^```json\s*|\s*```$', '', raw.strip())
    parsed = json.loads(cleaned)
    print("\nParsed JSON (after cleanup):")
    for key, value in parsed.items():
        print(f"  {key}: {value}")

### Pattern 3: Pydantic + tool_choice (Guaranteed Structure)

**When to use**: When you absolutely need guaranteed structure and type safety.

**Advantage**: Claude is *forced* to return the exact schema. No parsing failures. This is the most robust approach.

**How it works**: Define a Pydantic model, convert it to a tool definition, then force Claude to "call" that tool. The tool input IS your structured output.

In [ ]:
# === PATTERN 3: PYDANTIC + TOOL_CHOICE ===

from pydantic import BaseModel, Field
from typing import Literal

# Step 1: Define your output schema as a Pydantic model
class ReviewAnalysis(BaseModel):
    """Structured analysis of a product review."""
    sentiment: Literal["positive", "negative", "neutral", "mixed"] = Field(
        description="Overall sentiment of the review"
    )
    stars: int = Field(description="Star rating from 1 to 5", ge=1, le=5)
    key_topics: list[str] = Field(description="Main topics discussed in the review")
    summary: str = Field(description="One sentence summary of the review")
    would_recommend: bool = Field(description="Whether the reviewer would recommend the product")

# Step 2: Convert to tool definition
def pydantic_to_tool(model: type[BaseModel]):
    """Convert a Pydantic model to an Anthropic tool definition."""
    name = re.sub(r'(?<!^)(?=[A-Z])', '_', model.__name__).lower()
    return {
        "name": name,
        "description": model.__doc__ or "",
        "input_schema": model.model_json_schema()
    }

# Step 3: Call Claude with tool_choice forcing the tool
def extract_structured(user_input, model_class, system_prompt="Analyze the following input."):
    """Force Claude to return structured output matching the Pydantic model."""
    tool_def = pydantic_to_tool(model_class)
    
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=2048,
        system=system_prompt,
        tools=[tool_def],
        tool_choice={"type": "tool", "name": tool_def["name"]},
        messages=[{"role": "user", "content": user_input}]
    )
    
    # Extract the tool input — this IS our structured output
    tool_block = next(b for b in response.content if b.type == "tool_use")
    return model_class(**tool_block.input)

# Test it
review = "Terrible experience. The product arrived broken and customer support ghosted me. Never buying from this company again."

analysis = extract_structured(review, ReviewAnalysis, system_prompt="Analyze this product review.")

print(f"Type: {type(analysis).__name__}")
print(f"Sentiment: {analysis.sentiment}")
print(f"Stars: {analysis.stars}")
print(f"Topics: {analysis.key_topics}")
print(f"Summary: {analysis.summary}")
print(f"Would recommend: {analysis.would_recommend}")

### When to use which pattern:

| Pattern | Speed to implement | Reliability | Best for |
|---------|-------------------|-------------|----------|
| XML tags | Fastest (2 min) | High (95%+) | Interview demos, simple structures |
| JSON instruction | Fast (3 min) | Good (90%+) | When downstream code needs JSON |
| Pydantic + tool_choice | Medium (5 min) | Guaranteed (100%) | Production, complex schemas, type safety |

**Interview recommendation**: Start with XML tags for your first iteration. Upgrade to Pydantic + tool_choice if the interviewer cares about robustness.

---

## 4.6 Prompt Chaining Pattern

Use this when a task naturally breaks into sequential steps where each step's output feeds the next. Examples:
- Extract data, then analyze it, then write a summary
- Classify a ticket, then draft a response, then review the response
- Translate text, then localize it, then format it

In [ ]:
# === PROMPT CHAINING FUNCTION ===

def chain(steps, initial_input):
    """
    Execute a chain of prompts sequentially.
    
    steps: list of dicts with 'name', 'system_prompt', 'user_template'
           user_template should contain {input} placeholder
    initial_input: the starting input text
    
    Returns: dict mapping step name -> result
    """
    current_input = initial_input
    results = {}
    
    for step in steps:
        user_msg = step["user_template"].format(input=current_input)
        result = run(user_msg, system_prompt=step["system_prompt"])
        results[step["name"]] = result
        current_input = result
        print(f"[{step['name']}] done ({len(result)} chars)")
    
    return results

In [ ]:
# === DEMO: 3-step chain — Extract -> Analyze -> Respond ===

customer_email = """Subject: URGENT - API down for 6 hours

Hi Support,

I'm Marcus Rivera, CTO at DataFlow Inc (account #DF-4892). Our entire production 
pipeline has been down since 3am because your API keeps returning 503 errors on 
the /v2/transform endpoint. We process 2M records daily and we're already 6 hours 
behind.

We're on the Enterprise plan ($5,000/month) and our contract guarantees 99.9% 
uptime. This is the third outage this quarter.

I need:
1. Immediate acknowledgment and ETA for resolution
2. Root cause analysis within 24 hours
3. Discussion about SLA credits

If this isn't resolved by noon, we'll need to escalate to our account executive.

Marcus Rivera
marcus@dataflow.io
(555) 987-6543"""

steps = [
    {
        "name": "extract",
        "system_prompt": """Extract structured information from this customer email. Return:
<extraction>
<customer_name>NAME</customer_name>
<company>COMPANY</company>
<account_id>ID</account_id>
<plan>PLAN_TIER</plan>
<issue>BRIEF_DESCRIPTION</issue>
<severity>critical/high/medium/low</severity>
<requests>NUMBERED_LIST</requests>
<deadline>DEADLINE_IF_ANY</deadline>
</extraction>""",
        "user_template": "{input}"
    },
    {
        "name": "analyze",
        "system_prompt": """You are a customer success analyst. Given extracted customer data, assess the situation and recommend a response strategy.
Return:
<strategy>
<priority>P1/P2/P3</priority>
<churn_risk>high/medium/low</churn_risk>
<key_concerns>What matters most to this customer</key_concerns>
<recommended_actions>Specific steps to take, in order</recommended_actions>
<tone_guidance>How to communicate with this customer</tone_guidance>
</strategy>""",
        "user_template": "Analyze this customer situation and recommend a response strategy:\n\n{input}"
    },
    {
        "name": "respond",
        "system_prompt": """You are a senior customer support representative. Write a professional email response based on the provided strategy. Be empathetic, specific, and action-oriented. Include concrete next steps and timelines.""",
        "user_template": """Based on this analysis and strategy, write the customer response email:

{input}

The email should:
- Acknowledge the severity immediately
- Provide specific next steps with timelines
- Address each of their requests
- Be professional but empathetic
- Keep it under 200 words"""
    }
]

results = chain(steps, customer_email)

print("\n" + "=" * 60)
print("FINAL EMAIL RESPONSE:")
print("=" * 60)
print(results["respond"])

### Why chaining works well in interviews:

1. **Shows depth of thinking** -- you're not just throwing a prompt at the problem
2. **Each step is inspectable** -- you can show the intermediate results
3. **Easy to debug** -- if the output is wrong, you can find which step broke
4. **Modular** -- you can swap out or add steps without rewriting everything

---

## 4.7 When Things Go Wrong -- Recovery Patterns

Things will break during the live demo. Here are the quick fixes for the most common problems, with code you can copy-paste.

### Problem 1: Unexpected Output (Claude doesn't follow instructions)

In [ ]:
# === FIX: Add explicit constraints to the system prompt ===

# BEFORE (too vague):
vague_prompt = "Classify this support ticket."

# AFTER (specific constraints):
specific_prompt = """Classify this support ticket into exactly ONE of these categories: billing, technical, account, feature_request, general.

Rules:
- Return ONLY the category name, nothing else
- Do not explain your reasoning
- Do not include any other text
- Your entire response must be a single word from the list above"""

ticket = "The payment failed and now my account is locked."

print("Vague prompt result:")
print(run(ticket, system_prompt=vague_prompt))
print()
print("Specific prompt result:")
print(run(ticket, system_prompt=specific_prompt))

### Problem 2: Output Format is Wrong (JSON/XML not matching expected schema)

In [ ]:
# === FIX: Add an explicit example of the exact format ===

format_fix_prompt = """Classify this support ticket.

Return your response in EXACTLY this format (no deviations):

<result>
<category>CATEGORY_HERE</category>
<confidence>CONFIDENCE_HERE</confidence>
</result>

Example of a correct response:
<result>
<category>billing</category>
<confidence>high</confidence>
</result>

Valid categories: billing, technical, account, feature_request, general
Valid confidence levels: high, medium, low

Return ONLY the XML. No other text."""

print(run("My invoice shows the wrong amount.", system_prompt=format_fix_prompt))

### Problem 3: Claude Hallucinates (makes up information not in the input)

In [ ]:
# === FIX: Add grounding constraints ===

grounded_prompt = """You are a document analysis assistant.

CRITICAL RULES:
- Only use information that is EXPLICITLY stated in the provided text
- If a piece of information is not in the text, write "NOT_FOUND" for that field
- Do NOT infer, assume, or make up any information
- If you are unsure about something, say "UNCERTAIN" and explain why
- Quote the specific text that supports each extracted field

Extract the following from the provided text:
<extraction>
<name>NAME or NOT_FOUND</name>
<name_source>"exact quote from text" or N/A</name_source>
<company>COMPANY or NOT_FOUND</company>
<company_source>"exact quote from text" or N/A</company_source>
<budget>BUDGET or NOT_FOUND</budget>
<budget_source>"exact quote from text" or N/A</budget_source>
</extraction>"""

# Test with a message that has incomplete info
sparse_input = "Hey, this is Mike. We need the integration done by Friday."
print(run(sparse_input, system_prompt=grounded_prompt))

### Problem 4: Response is Too Long or Too Short

In [ ]:
# === FIX: Add explicit length constraints ===

# Too long? Add a ceiling:
concise_prompt = """Summarize the following text.

Length constraint: Your summary must be EXACTLY 2-3 sentences. No more than 50 words total.
Do not include any preamble like 'Here is a summary'. Start directly with the summary."""

# Too short? Add a floor:
detailed_prompt = """Analyze the following text in detail.

Your analysis must include:
- At least 3 key findings (one sentence each)
- At least 2 action items
- A risk assessment paragraph (3-4 sentences minimum)

Do not be brief. Provide thorough, detailed analysis."""

test_text = "Q3 revenue was $2.4M, up 15% from Q2. Customer churn increased to 8%. Three enterprise deals are in the pipeline worth a combined $500K."

print("=== CONCISE VERSION ===")
print(run(test_text, system_prompt=concise_prompt))
print()
print("=== DETAILED VERSION ===")
print(run(test_text, system_prompt=detailed_prompt))

### Problem 5: Claude Refuses the Task

In [ ]:
# === FIX: Reframe with clear, benign context ===

# If Claude refuses, add context about WHY this task is being done:
reframed_prompt = """You are a content moderation training system for a children's online safety nonprofit.

Your job is to classify user-generated messages by risk level so human moderators 
can prioritize their review queue. This system PROTECTS children by flagging 
concerning content for human review.

Classify each message as:
- safe: Normal, appropriate content
- review: Potentially concerning, needs human review
- urgent: Requires immediate human moderator attention

<classification>
<risk_level>safe/review/urgent</risk_level>
<reasoning>Brief explanation</reasoning>
</classification>

Remember: You are not making final decisions. You are helping human moderators 
prioritize their work to keep children safe."""

print(run("Hey everyone, want to play Minecraft after school?", system_prompt=reframed_prompt))

### Quick Recovery Cheat Sheet

| Problem | Fix | Time to implement |
|---------|-----|-------------------|
| Unexpected output | Add explicit constraints and rules | 30 sec |
| Wrong format | Add example of exact expected output | 30 sec |
| Hallucination | Add "only use provided info" + source quotes | 30 sec |
| Too long/short | Add word count or sentence count constraints | 15 sec |
| Refusal | Reframe with benign context, explain the purpose | 1 min |
| Inconsistent results | Add few-shot examples (Section 4.3) | 2 min |
| Edge case failure | Add specific handling rule in system prompt | 30 sec |

---

## 4.8 The Demo Presentation Flow

When it's time to present your solution to the "customer" (the interviewer), follow this structure. It shows you think like a solutions engineer, not just a coder.

### Step 1: Recap (30 seconds)

> "So what I heard is that you need [restate the problem in your own words]. 
> The key requirements are [list 2-3 must-haves]. Is that right?"

This shows you listened and gives them a chance to correct any misunderstanding.

### Step 2: Approach (1 minute)

> "Here's how I approached this:
> - I used [technique] because [reason]
> - The system prompt handles [X] by [Y]
> - I chose [structured output format] because [reason]"

Briefly explain your design choices. Mention alternatives you considered.

### Step 3: Live Demo (3-5 minutes)

Run `quick_eval` with 3-5 diverse test cases. Walk through each result:

> "Let me show you this working on some representative inputs..."
> 
> [Run the eval]
> 
> "For this first case, notice how it correctly identifies [X]..."
> "This second case is interesting because [edge case or nuance]..."

### Step 4: Edge Cases (1 minute)

> "I also tested some tricky edge cases:
> - [Ambiguous input] -- it handles this by [X]
> - [Missing data] -- it returns 'unknown' rather than guessing
> - [Adversarial input] -- the constraints prevent [unwanted behavior]"

### Step 5: Limitations (1 minute)

> "Here's where this approach might struggle:
> - [Limitation 1] -- I'd address this with [mitigation]
> - [Limitation 2] -- for production, we'd want [improvement]"

Being upfront about limitations shows maturity. Always pair a limitation with a mitigation.

### Step 6: Next Steps (1 minute)

> "For production, I'd add:
> - An evaluation suite with 50+ test cases
> - Prompt caching for cost optimization
> - Human-in-the-loop for uncertain cases
> - Monitoring and alerting for output quality"

This is where Section 4.9 talking points come in.

---

## 4.9 Production Talking Points

These are things to mention that show you think beyond the prototype. You don't need to implement these -- just mentioning them demonstrates depth.

### Cost Optimization

- **Prompt caching**: "For repeated system prompts, we'd enable prompt caching to reduce latency by up to 85% and cost by up to 90% on cached tokens. The system prompt and few-shot examples would be cached across calls."
  
- **Batches API**: "For batch processing scenarios like processing 500 applications per cycle, we'd use the Batches API for 50% cost savings. Results come back within 24 hours, which is fine for non-real-time workflows."

- **Model selection**: "We'd tier the workload by complexity:
  - **Haiku** for simple classification at scale (fast, cheap, accurate for straightforward tasks)
  - **Sonnet** for this level of nuance and complexity (best price-to-performance)
  - **Opus** for ambiguous cases that need deep reasoning (most capable, use sparingly)"

### Quality Assurance

- **Evaluation**: "I'd build an eval suite with 50+ test cases covering normal cases, edge cases, and adversarial inputs. Measure with exact match for classification, and LLM-as-judge for generation quality. Track consistency across runs."

- **Human-in-the-loop**: "For high-stakes decisions (e.g., grant approvals, content moderation), Claude should flag uncertain cases for human review. Set a confidence threshold -- anything below 'high' confidence goes to a human queue."

- **Monitoring**: "In production, I'd log all inputs and outputs, track output quality metrics over time, and set up alerts for distribution shifts (e.g., suddenly classifying 80% of tickets as 'billing' when the historical rate is 30%)."

### Robustness

- **Retry logic**: "Add exponential backoff for API errors. 429 (rate limit) and 529 (overloaded) are retryable. 400 (bad request) is not."

- **Fallback models**: "If Sonnet is down or slow, fall back to Haiku with an adjusted prompt. For critical paths, consider multi-model consensus."

- **Input validation**: "Validate and sanitize inputs before sending to the API. Set max input length, strip PII if needed, reject obviously malicious inputs."

### Architecture

- **Prompt versioning**: "Version control all prompts. A/B test prompt changes before rolling out. Never change a production prompt without running the eval suite."

- **Streaming**: "For user-facing applications, use streaming responses so the user sees output immediately rather than waiting for the full response."

- **Caching layer**: "For deterministic queries (same input = same expected output), cache responses to avoid redundant API calls."

---

## Summary: The Speed Run Checklist

When the timer starts, do these in order:

### 1. Discovery (10 min) -- Ask questions, take notes
- What is the input? (format, length, language)
- What is the desired output? (format, structure, examples)
- What are the edge cases? (ambiguous inputs, missing data, adversarial)
- What matters most? (accuracy, speed, cost, consistency)
- What's the volume? (real-time vs batch, how many per day)

### 2. Setup (1 min) -- Imports, client, `run()` function
```python
!pip install anthropic -q
import anthropic, json, re
client = anthropic.Anthropic()
def run(msg, sys="You are a helpful assistant."):
    return client.messages.create(model="claude-haiku-4-5-20251001", max_tokens=4096,
        system=sys, messages=[{"role":"user","content":msg}]).content[0].text
```

### 3. System Prompt (3 min) -- Pick the right template
- Classification task? -> Template A (The Classifier)
- Data extraction task? -> Template B (The Extractor)
- Content generation? -> Template C (The Generator)
- Analysis/summary? -> Template D (The Analyst)

### 4. First Test (2 min) -- One input, see what comes out
```python
print(run("test input here", system_prompt))
```
Does the format look right? Is the content reasonable?

### 5. Iterate (5 min) -- Improve based on what you see
- Add few-shot examples if output format is inconsistent
- Add constraints if output has unwanted content
- Add grounding rules if Claude is hallucinating
- Use Pydantic + tool_choice if you need guaranteed structure

### 6. Quick Eval (3 min) -- Test 3-5 diverse cases
```python
test_cases = [
    {"input": "normal case", "expected": "expected_output"},
    {"input": "edge case", "expected": "expected_output"},
    {"input": "tricky case", "expected": "expected_output"},
]
quick_eval(system_prompt, test_cases, parse_fn=my_parser)
```

### 7. Prepare Demo (1 min) -- Organize and rehearse
- Clean up your notebook (delete failed experiments)
- Note 2-3 talking points about your design choices
- Note 1-2 limitations and how you'd address them
- Note 2-3 production considerations (Section 4.9)

**Total: ~25 minutes of the 30-minute build window, leaving 5 minutes of buffer.**

---

**Remember**: The interviewer is evaluating your problem-solving process, not just the final output. Think out loud, explain your choices, and show that you can iterate quickly when something doesn't work.